In [ ]:
# Baseline Inference Code for PYNQ

import numpy as np
import json
import time

# Load artifacts (pure NumPy with no torch required)
params = {
    "fc1": {
        "weight_int8": np.load("artifacts/fc1_weight.npy"),
        "bias_fp32":   np.load("artifacts/fc1_bias.npy"),
    },
    "fc2": {
        "weight_int8": np.load("artifacts/fc2_weight.npy"),
        "bias_fp32":   np.load("artifacts/fc2_bias.npy"),
    },
    "fc3": {
        "weight_int8": np.load("artifacts/fc3_weight.npy"),
        "bias_fp32":   np.load("artifacts/fc3_bias.npy"),
    },
}

with open("artifacts/activation_scales.json") as f:
    act_scales = json.load(f)

with open("artifacts/weight_scales.json") as f:
    weight_scales = json.load(f)

# Load MNIST test set
images = np.load("artifacts/mnist_test_images.npy")  # (10000, 28, 28), float32 [0,1]
labels = np.load("artifacts/mnist_test_labels.npy")  # (10000,)


# Inference primitives
def quantize_to_int8(x_fp32, scale):
    return np.clip(np.round(x_fp32 / scale), -128, 127).astype(np.int8)

def linear_int8(x_int8, w_int8, x_scale, w_scale, bias_fp32):
    y_int32 = x_int8.astype(np.int32) @ w_int8.T.astype(np.int32)
    y_fp32  = y_int32.astype(np.float32) * (x_scale * w_scale)
    if bias_fp32 is not None and bias_fp32.size > 0:
        y_fp32 += bias_fp32.reshape(1, -1)
    return y_fp32

def relu(x):
    return np.maximum(x, 0.0).astype(np.float32)

def softmax(logits):
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_x  = np.exp(logits)
    return (exp_x / np.sum(exp_x, axis=1, keepdims=True)).astype(np.float32)


# Single-sample inference
def predict_one(image_fp32):
    x = image_fp32.reshape(1, -1)

    x_int8 = quantize_to_int8(x, act_scales["input"])

    fc1 = linear_int8(x_int8, params["fc1"]["weight_int8"],
                      act_scales["input"], weight_scales["fc1"],
                      params["fc1"]["bias_fp32"])
    x_int8 = quantize_to_int8(relu(fc1), act_scales["relu1_out"])

    fc2 = linear_int8(x_int8, params["fc2"]["weight_int8"],
                      act_scales["relu1_out"], weight_scales["fc2"],
                      params["fc2"]["bias_fp32"])
    x_int8 = quantize_to_int8(relu(fc2), act_scales["relu2_out"])

    fc3 = linear_int8(x_int8, params["fc3"]["weight_int8"],
                      act_scales["relu2_out"], weight_scales["fc3"],
                      params["fc3"]["bias_fp32"])

    probs = softmax(fc3)
    return int(np.argmax(probs, axis=1)[0]), fc3[0], probs[0]


# Sample predictions
print("\n===== Sample Predictions =====")
for idx in [0, 1, 2, 3, 4]:
    pred, logits, probs = predict_one(images[idx])
    print(f"Sample {idx}: label={labels[idx]}, pred={pred}")
    print(f"  logits = {np.round(logits, 4)}")
    print(f"  probs  = {np.round(probs,  4)}")

# Accuracy on first 1000 samples
print("\n===== Accuracy =====")
correct = sum(predict_one(images[i])[0] == labels[i] for i in range(1000))
print(f"Accuracy on first 1000 samples: {correct / 1000:.4f}")

# Latency benchmark
print("\n===== Timing =====")
start = time.perf_counter()
for i in range(1000):
    predict_one(images[i])
avg_ms = (time.perf_counter() - start) * 1000.0 / 1000
print(f"Average inference latency: {avg_ms:.4f} ms/sample")

In [ ]:
# FPGA-Accelerated Inference for PYNQ

import numpy as np
import json
from pynq import Overlay, allocate

# Load quantized artifacts (pure NumPy with no torch required)
w_fc1 = np.load("artifacts/fc1_weight.npy") # shape (128, 784), int8
b_fc1 = np.load("artifacts/fc1_bias.npy") # shape (128,), float32
with open("artifacts/weight_scales.json") as f:
    w_scale_fc1 = json.load(f)["fc1"]
with open("artifacts/activation_scales.json") as f:
    act_scales = json.load(f)

# Load overlay
FC1_M = 1
FC1_K = 784
FC1_N = 128

ol = Overlay("block_design.bit") # corret .bit name

ip = ol.multiply_0 # corrent IP core instance name

# Allocate DMA-capable buffers
A_buf = allocate(shape=(FC1_M * FC1_K,), dtype=np.int8)
B_buf = allocate(shape=(FC1_K * FC1_N,), dtype=np.int8)
C_buf = allocate(shape=(FC1_M * FC1_N,), dtype=np.int32)

# Load one MNIST sample
images = np.load("artifacts/mnist_test_images.npy") # shape (10000, 28, 28), float32 [0,1]
labels = np.load("artifacts/mnist_test_labels.npy") # shape (10000,)
label = int(labels[0])

x_fp32 = images[0].reshape(-1) # shape (784,)
x_int8 = np.clip(np.round(x_fp32 / act_scales["input"]), -128, 127).astype(np.int8)

# Fill buffers
A_buf[:] = x_int8
B_buf[:] = w_fc1.T.flatten() # transpose to (K, N) = (784, 128) row-major

A_buf.sync_to_device()
B_buf.sync_to_device()

# Configure IP registers
ip.write(0x10, A_buf.device_address & 0xFFFFFFFF)
ip.write(0x14, (A_buf.device_address >> 32) & 0xFFFFFFFF)
ip.write(0x1c, B_buf.device_address & 0xFFFFFFFF)
ip.write(0x20, (B_buf.device_address >> 32) & 0xFFFFFFFF)
ip.write(0x28, C_buf.device_address & 0xFFFFFFFF)
ip.write(0x2c, (C_buf.device_address >> 32) & 0xFFFFFFFF)
ip.write(0x34, FC1_M)
ip.write(0x3c, FC1_K)
ip.write(0x44, FC1_N)

# Start and poll
ip.write(0x00, 0x01)

while (ip.read(0x00) & 0x2) == 0:
    pass

# Retrieve and dequantize
C_buf.sync_from_device()

y_int32 = np.array(C_buf).reshape(FC1_M, FC1_N) # (1, 128) int32
y_fp32  = y_int32.astype(np.float32) * (act_scales["input"] * w_scale_fc1)
y_fp32 += b_fc1 # add bias in float
y_relu  = np.maximum(y_fp32, 0.0) # ReLU then feed to FC2

print(f"Label: {label}, FC1 output (first 8): {y_relu[0, :8]}")

A_buf.close()
B_buf.close()
C_buf.close()

In [ ]:
# Compare FPGA and CPU Results

import numpy as np
import json

# Reload artifacts
params = {
    "fc1": {
        "weight_int8": np.load("artifacts/fc1_weight.npy"),
        "bias_fp32":   np.load("artifacts/fc1_bias.npy"),
    },
}
with open("artifacts/activation_scales.json") as f:
    act_scales = json.load(f)
with open("artifacts/weight_scales.json") as f:
    weight_scales = json.load(f)

images = np.load("artifacts/mnist_test_images.npy")
x = images[0].reshape(1, -1)

# Software FC1
x_int8 = np.clip(np.round(x / act_scales["input"]), -128, 127).astype(np.int8)
y_int32 = x_int8.astype(np.int32) @ params["fc1"]["weight_int8"].T.astype(np.int32)
y_fp32  = y_int32.astype(np.float32) * (act_scales["input"] * weight_scales["fc1"])
y_fp32 += params["fc1"]["bias_fp32"]
y_relu_sw = np.maximum(y_fp32, 0.0)

print("Software FC1 (first 8):", y_relu_sw[0, :8])
print("FPGA    FC1 (first 8):", y_relu[0, :8])
print("Max diff:", np.max(np.abs(y_relu_sw[0] - y_relu[0])))

In [ ]:
# Full FPGA-Accelerated Batched Inference (FC1 → FC2 → FC3)

import numpy as np
import json
import time
from pynq import Overlay, allocate

BATCH = 32 # configurable number of images per GEMM call

# Load all artifacts
w = {
    "fc1": np.load("artifacts/fc1_weight.npy"),  # (128, 784) int8
    "fc2": np.load("artifacts/fc2_weight.npy"),  # (64, 128)  int8
    "fc3": np.load("artifacts/fc3_weight.npy"),  # (10, 64)   int8
}
b = {
    "fc1": np.load("artifacts/fc1_bias.npy"),
    "fc2": np.load("artifacts/fc2_bias.npy"),
    "fc3": np.load("artifacts/fc3_bias.npy"),
}
with open("artifacts/weight_scales.json") as f:
    ws = json.load(f)
with open("artifacts/activation_scales.json") as f:
    act_scales = json.load(f)

images = np.load("artifacts/mnist_test_images.npy")
labels = np.load("artifacts/mnist_test_labels.npy")

ol = Overlay("block_design.bit")
ip = ol.multiply_0

# Pre-load weight buffers once — never copied again during inference
layers = [
    {"name": "fc1", "K": 784, "N": 128},
    {"name": "fc2", "K": 128, "N":  64},
    {"name": "fc3", "K":  64, "N":  10},
]

for layer in layers:
    K, N = layer["K"], layer["N"]
    buf = allocate(shape=(K * N,), dtype=np.int8)
    buf[:] = w[layer["name"]].T.flatten() # transpose to (K, N) row-major
    buf.sync_to_device() # sync once, never again
    layer["B_buf"] = buf

# A and C buffers sized for FC1 at full batch size
A_buf = allocate(shape=(BATCH * 784,), dtype=np.int8)
C_buf = allocate(shape=(BATCH * 128,), dtype=np.int32)


def run_fpga_gemm(x_int8, B_buf, M, K, N):
    # Run one batched GEMM
    # A=(M,K), B=(K,N), C=(M,N)
    A_buf[:M * K] = x_int8.reshape(-1)
    A_buf.sync_to_device()

    ip.write(0x10, A_buf.device_address & 0xFFFFFFFF)
    ip.write(0x14, (A_buf.device_address >> 32) & 0xFFFFFFFF)
    ip.write(0x1c, B_buf.device_address & 0xFFFFFFFF)
    ip.write(0x20, (B_buf.device_address >> 32) & 0xFFFFFFFF)
    ip.write(0x28, C_buf.device_address & 0xFFFFFFFF)
    ip.write(0x2c, (C_buf.device_address >> 32) & 0xFFFFFFFF)
    ip.write(0x34, M)
    ip.write(0x3c, K)
    ip.write(0x44, N)

    ip.write(0x00, 0x01)
    while (ip.read(0x00) & 0x2) == 0:
        pass

    C_buf.sync_from_device()
    return np.array(C_buf[:M * N]).reshape(M, N)


def quantize(x_fp32, scale):
    return np.clip(np.round(x_fp32 / scale), -128, 127).astype(np.int8)

def dequantize_bias_relu(y_int32, x_scale, w_scale, bias):
    y = y_int32.astype(np.float32) * (x_scale * w_scale) + bias
    return np.maximum(y, 0.0).astype(np.float32)

def softmax(logits):
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_x = np.exp(logits)
    return (exp_x / np.sum(exp_x, axis=1, keepdims=True)).astype(np.float32)


def predict_batch(images_fp32):
    # Run inference on a batch of images
    # Returns array of BATCH predictions
    M = images_fp32.shape[0]

    # FC1: A=(M,784), B=(784,128) -> C=(M,128)
    x_int8 = quantize(images_fp32.reshape(M, -1), act_scales["input"])
    fc1 = run_fpga_gemm(x_int8, layers[0]["B_buf"], M=M, K=784, N=128)
    x_int8 = quantize(dequantize_bias_relu(fc1, act_scales["input"], ws["fc1"], b["fc1"]),
                      act_scales["relu1_out"])

    # FC2: A=(M,128), B=(128,64) -> C=(M,64)
    fc2 = run_fpga_gemm(x_int8, layers[1]["B_buf"], M=M, K=128, N=64)
    x_int8 = quantize(dequantize_bias_relu(fc2, act_scales["relu1_out"], ws["fc2"], b["fc2"]),
                      act_scales["relu2_out"])

    # FC3: A=(M,64), B=(64,10) -> C=(M,10)
    fc3 = run_fpga_gemm(x_int8, layers[2]["B_buf"], M=M, K=64, N=10)
    logits = fc3.astype(np.float32) * (act_scales["relu2_out"] * ws["fc3"]) + b["fc3"]

    probs = softmax(logits)
    return np.argmax(probs, axis=1), logits, probs


# Sample predictions (first 5 images as a batch)
print(f"===== FPGA Batched Predictions (batch={BATCH}) =====")
preds, logits, probs = predict_batch(images[:BATCH])
for idx in range(5):
    print(f"Sample {idx}: label={labels[idx]}, pred={preds[idx]}")
    print(f"  logits = {np.round(logits[idx], 4)}")
    print(f"  probs  = {np.round(probs[idx],  4)}")

# Accuracy on first 1000 samples
print("\n===== FPGA Accuracy =====")
correct = 0
for i in range(0, 1000, BATCH):
    batch_imgs = images[i:i+BATCH]
    batch_preds, _, _ = predict_batch(batch_imgs)
    correct += np.sum(batch_preds == labels[i:i+BATCH])
print(f"Accuracy on first 1000 samples: {correct / 1000:.4f}")

# Latency benchmark
print("\n===== FPGA Timing =====")
n_batches = 100 // BATCH  # run enough batches to cover 100 samples
start = time.perf_counter()
for i in range(n_batches):
    predict_batch(images[i*BATCH:(i+1)*BATCH])
total_samples = n_batches * BATCH
avg_ms = (time.perf_counter() - start) * 1000.0 / total_samples
print(f"Batch size:                {BATCH}")
print(f"Average FPGA latency:      {avg_ms:.4f} ms/sample")

# Cleanup
A_buf.close()
C_buf.close()
for layer in layers:
    layer["B_buf"].close()
